In [0]:
from pyspark.sql.functions import col , current_timestamp,regexp_extract, split, date_format , to_date

In [0]:
landing_path    = "/Volumes/skypoints/landing/members"
checkpoint_path = "/Volumes/skypoints/bronze/members/member_checkpoints"


In [0]:
file_schema = """
    filler            STRING,   -- empty text before the leading pipe
    record_type       STRING,   -- H = header, D = data
    member_name       STRING,
    member_id         STRING,
    enrollment_date   STRING,
    last_flight_date  STRING,
    tier              STRING,
    agent_name        STRING,
    state             STRING,
    country           STRING,
    post_code         STRING,
    dob               STRING,
    is_active         STRING,
    _rescued_data     STRING
"""

In [0]:
%sql
CREATE TABLE IF NOT EXISTS skypoints.bronze.raw_member
(
    -- fields from the member file, all raw text
    member_id          STRING,
    member_name        STRING,
    country            STRING,
    dob                STRING COMMENT 'Text on purpose: 03051985 must keep its leading zero',
    enrollment_date    STRING,
    last_flight_date   STRING,
    tier               STRING,
    is_active          STRING,
    agent_name         STRING,
    state              STRING,
    post_code          STRING,

    -- audit
     _rescued_data     STRING,
    source_file_name   STRING    COMMENT 'e.g. SKYPOINTS_MEMBERS_20240115_12000000.txt',
    file_date          DATE      COMMENT 'Parsed from the file name',
    ingestion_timestamp TIMESTAMP COMMENT 'When the row was loaded into bronze'
    
)
USING DELTA
COMMENT 'Bronze: raw member rows, append-only, no casting';

In [0]:


df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("delimiter", "|")
    .option("header", "false")                     
    .option("rescuedDataColumn", "_rescued_data")
    .schema(file_schema)
    .load(landing_path)
    .filter(col("record_type") == "D")
    .drop("filler", "record_type")
    .withColumn("source_file_name", col("_metadata.file_name"))
    .withColumn("file_date", to_date(regexp_extract(col("_metadata.file_name"), r"_(\d{8})_", 1), "yyyyMMdd"))
    .withColumn("ingestion_timestamp", current_timestamp())

)

In [0]:
(
    df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)                   # process what is there, then stop
    .toTable("skypoints.bronze.raw_member")
)

In [0]:
%sql
select count(*) from skypoints.bronze.raw_member
where member_name IS NULL;